<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/stm_deletion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -U langchain-openai

In [22]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import RemoveMessage

In [23]:
api_key = userdata.get("OPENAI_API_KEY")
model = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

In [24]:
def call_model(state: MessagesState):
  response = model.invoke(state["messages"])
  return {"messages": [response]}

In [25]:
def delete_old_messages(state: MessagesState):
  messages = state["messages"]

  # if messages > 10, delete the initial 6 messages
  if len(messages) > 10:
    to_remove = messages[:6]
    print("-"*80)
    print("Deleted initial 6 messages.")
    print("-"*80)
    return {"messages": [RemoveMessage(id=m.id) for m in to_remove]}

  return {}

In [26]:
graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)
graph.add_node("delete_old_messages", delete_old_messages)

graph.add_edge(START, "call_model")
graph.add_edge("call_model", "delete_old_messages") # run cleanup after each response
graph.add_edge("delete_old_messages", END)

In [27]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [28]:
config = {"configurable": {"thread_id": "thread-1"}}
workflow.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config=config)
workflow.invoke({"messages": [{"role": "user", "content": "I'm agent 007."}]}, config=config)
workflow.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config)
workflow.invoke({"messages": [{"role": "user", "content": "Am i relevant in this AI agent era?"}]}, config=config)
workflow.invoke({"messages": [{"role": "user", "content": "I'm learning about LangChain and LangGraph."}]}, config=config)
workflow.invoke({"messages": [{"role": "user", "content": "What are LLM memory? And explain short term memory thoroughly with example?"}]}, config=config)

--------------------------------------------------------------------------------
Deleted initial 6 messages.
--------------------------------------------------------------------------------


{'messages': [HumanMessage(content='Am i relevant in this AI agent era?', additional_kwargs={}, response_metadata={}, id='e6d62ed5-acc8-4a14-bfa3-7c94f3626114'),
  AIMessage(content='Absolutely, James. AI agents can automate tasks, but people still set goals, judge what matters, build trust, and take responsibility. The future isn’t just about who can do the most—it’s about who knows what’s worth doing.\n\nEven 007 needs a human behind the mission.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 98, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 71, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ElG6flcNCAZHsiaweDyIGktecrUBW', 'service_tier': 'default', 'finish_reason

In [30]:
graph_state = workflow.get_state(config=config)
values = graph_state.values

print("-"*80)
print("No. of messages in state: ", len(values.get("messages", [])))
print("-"*80)

for message in values.get("messages", []):
  print(type(message).__name__, ": ", message.content)

--------------------------------------------------------------------------------
No. of messages in state:  6
--------------------------------------------------------------------------------
HumanMessage :  Am i relevant in this AI agent era?
AIMessage :  Absolutely, James. AI agents can automate tasks, but people still set goals, judge what matters, build trust, and take responsibility. The future isn’t just about who can do the most—it’s about who knows what’s worth doing.

Even 007 needs a human behind the mission.
HumanMessage :  I'm learning about LangChain and LangGraph.
AIMessage :  That’s a useful area to explore, James. **LangChain** helps connect language models to tools, data, and other components; **LangGraph** is especially useful for building stateful, multi-step agent workflows with branching and human oversight.

A practical learning path:
1. Build a simple tool-using assistant with LangChain.
2. Rebuild it in LangGraph with explicit state and conditional steps.
3. Add 